# RestAtLastTradePrice 三日全市场审计

## tl;dr

当前三日全量运行尚未完成；不能报告全部通过。20260828 冒烟 SH 28,225、SZ 26,401 帧匹配，均无不匹配或数据错误。最终结论必须以六个 full 任务全部结束后的审计为准。

## Context & Methods

范围为 20260828、20260601、20260806，分别恢复 SH/SZ 全部支持的股票和 ETF。读取 `/hdd/data/stock/raw_level2_parquet` 的逐笔与 raw snapshot，不使用 canonical snapshot。CLI `validate` 执行整日恢复，按规范比较开盘后、盘中及收盘状态，不生成全市场固定间隔文件。

### Key Assumptions

- SZ 市价单采用 `rest_at_last_trade_price`；本方最优仍保留空盘口严格撤单对账。策略不保证每个中间态／FIFO 路径，`standard_acceptance=false` 是预期标记。
- 比较字段及窗口不放宽；SH ETF 以 20260706 为收盘阶段规则分界。
- `matched/(matched+mismatched)` 为可比帧匹配率；状态排除、数据错误、缺少来源分列。回放中止没有全日匹配率。
- 日期池抽取在运行前完成；不得根据结果更换失败日期。每个日期两个市场均执行，不将冒烟计入全量结果。
- 本 notebook 仅审计已有结果，不重扫原始全量数据。重跑任务入口为 `analysis/run_practical_full_validation.py --output <新目录>`。

## Data

### 1. 加载来源清单与独立审计

仅需 Python 标准库。可从项目根目录或 `analysis/` 运行。运行尚未完成时允许显示进度；最终验收请将 `REQUIRE_COMPLETE` 改为 `True`。

In [1]:
from pathlib import Path
import json, runpy
ROOT = Path.cwd()
if not (ROOT / 'Cargo.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'Cargo.toml').is_file()
OUT = ROOT / 'reports/20260906-rest-at-last-trade-full'
REQUIRE_COMPLETE = False
manifest = json.loads((OUT / 'manifest.json').read_text())
audit_fn = runpy.run_path(str(ROOT / 'analysis/audit_practical_validation.py'))['audit']
result = audit_fn(OUT, require_complete=REQUIRE_COMPLETE)
print('binary SHA256:', result['binary_sha256'])
print('source partitions:', len(manifest['sources']), 'dates:', manifest['days'])
for day in manifest['days']:
    print(day, {source['feed']: source['rows'] for source in manifest['sources'] if source['date'] == day})


binary SHA256: b78c3270762e1b0f5e2deeb1eef40a54d03b43015b55c599291049b4c77f0ed9
source partitions: 15 dates: ['20260828', '20260601', '20260806']
20260828 {'mdl_4_24_0': 208856991, 'MarketData': 14272071, 'mdl_6_33_0': 152645817, 'mdl_6_36_0': 137935478, 'mdl_6_28_0': 15722868}
20260601 {'mdl_4_24_0': 242093440, 'MarketData': 17426623, 'mdl_6_33_0': 178932534, 'mdl_6_36_0': 163061290, 'mdl_6_28_0': 16119011}
20260806 {'mdl_4_24_0': 238205940, 'MarketData': 14566747, 'mdl_6_33_0': 177974699, 'mdl_6_36_0': 161989323, 'mdl_6_28_0': 16050997}


## Results

### 2. 任务完整性与分母

审计断言包括：冻结二进制一致、全量命令没有 `--symbols`、行情输入行数等于来源 footer、各分类加总一致、无诊断窗口覆盖。

In [2]:
print('all_jobs_finished:', result['all_jobs_finished'])
print('all_snapshot_validations_passed:', result['all_snapshot_validations_passed'])
for row in result['rows']:
    keys = ['date', 'market', 'status', 'exit_code', 'matched', 'mismatched',
            'excluded_by_status', 'data_errors', 'missing_source', 'comparable_match_rate',
            'replayed_symbols', 'applied_events', 'pending_groups', 'empty_same_side_cancellations']
    print({key: row[key] for key in keys if key in row})


all_jobs_finished: False
all_snapshot_validations_passed: False
{'date': '20260828', 'market': 'SH', 'status': 'running'}
{'date': '20260828', 'market': 'SZ', 'status': 'running'}
{'date': '20260601', 'market': 'SH', 'status': 'not_started'}
{'date': '20260601', 'market': 'SZ', 'status': 'not_started'}
{'date': '20260806', 'market': 'SH', 'status': 'not_started'}
{'date': '20260806', 'market': 'SZ', 'status': 'not_started'}


### 3. 品种、阶段和异常分布

聚合数据完整；失败明细最多 5000 条，不能用明细条数代替总异常数。

In [3]:
for row in result['rows']:
    if 'breakdown' not in row:
        if 'error' in row:
            print(row['date'], row['market'], 'ABORT:', row['error'][-1500:])
        continue
    print(row['date'], row['market'])
    for group, counts in sorted(row['breakdown'].items()):
        denominator = counts['comparable']
        rate = counts['matched'] / denominator if denominator else None
        print(group, 'matched/comparable', counts['matched'], '/', denominator,
              'match_rate', rate, 'excluded/errors/missing',
              counts['excluded_by_status'], counts['data_errors'], counts['missing_source'])
    print('mismatch fields:', row['mismatch_fields'])
    print('not comparable reasons:', row['not_comparable_reasons'])


## Takeaways

六个 full 任务尚未全部结束，本次 notebook 只构成运行完整性检查，不构成最终通过结论。更新结果后重新从头执行；只有所有 full 运行成功、数据与缺失来源错误为零，才可声称本次范围的 snapshot 全匹配。即便全部匹配，也不证明实用市价算法所有逐事件中间态正确。